# Module Speech-to-Text (STT) avec Whisper

Ce notebook implémente la partie **Parole → Texte** du projet *Voix-langue-des-signes*.

Objectif :
- Enregistrer la voix d’un utilisateur à partir du micro.
- Transcrire l’audio en texte (français) à l’aide du modèle Whisper d’OpenAI.
- Préparer une fonction réutilisable (`speech_to_text`) que les autres parties du projet pourront appeler.

Ce notebook est utilisé dans la branche `feature/stt-yohann` du dépôt Git.


## 1. Configuration de `ffmpeg`

Whisper s’appuie sur `ffmpeg` pour charger et décoder les fichiers audio (WAV, MP3, etc.).

Sur notre machine Windows, nous avons :
1. Téléchargé **ffmpeg** depuis le site : https://www.gyan.dev/ffmpeg/builds/
2. Choisi l’archive `ffmpeg-release-essentials.zip`
3. Extrait le contenu dans le dossier : `C:\ffmpeg\ffmpeg-8.0-essentials_build\`
4. Ajouté le chemin `C:\ffmpeg\ffmpeg-8.0-essentials_build\bin` à la variable d’environnement `PATH` de Windows.

Cependant, le kernel Python utilisé par VS Code ne récupère pas toujours correctement le `PATH` système.
Nous ajoutons donc explicitement ce chemin dans le `PATH` du **processus Python** dans ce notebook.


In [2]:
import os
import shutil

# On ajoute le dossier ffmpeg au PATH du processus Python
os.environ["PATH"] += r";C:\ffmpeg\ffmpeg-8.0-essentials_build\bin"

print("ffmpeg trouvé par Python ? ->", shutil.which("ffmpeg"))


ffmpeg trouvé par Python ? -> C:\ffmpeg\ffmpeg-8.0-essentials_build\bin\ffmpeg.EXE


## 2. Installation des dépendances Python

Les principales bibliothèques utilisées sont :
- `whisper` : modèle de reconnaissance vocale pré-entraîné.
- `sounddevice` : enregistrement audio depuis le micro.
- `scipy` + `numpy` : manipulation et sauvegarde des signaux audio.

Sur cette machine, ces paquets ont été installés via `pip` :

```bash
pip install openai-whisper sounddevice scipy


In [ ]:
# À exécuter seulement si les libs ne sont pas encore installées sur la machine
# !pip install openai-whisper sounddevice scipy --quiet

## 3. Imports et configuration de base

Dans cette cellule, nous :
- importons les modules nécessaires (`whisper`, `sounddevice`, `numpy`, etc.),
- affichons le dossier de travail courant,
- listons les fichiers présents (utile pour vérifier la présence de `mon_audio.wav` après enregistrement).


In [3]:
import whisper
import sounddevice as sd
from scipy.io.wavfile import write
import numpy as np

import os

print("Dossier courant :", os.getcwd())
print("Fichiers présents :", os.listdir())


Dossier courant : d:\Desktop\voix-langue-des-signes\stt
Fichiers présents : ['mon_audio.wav', 'stt_speech_to_text.ipynb']


## 4. Chargement du modèle Whisper

Nous utilisons le modèle **`small`** de Whisper, qui offre un bon compromis entre :
- temps de calcul (raisonnable sur CPU),
- qualité de la transcription.

Remarque :
- Le warning `FP16 is not supported on CPU; using FP32 instead` est **normal** sur CPU :
  cela signifie simplement que le modèle utilisera des flottants 32 bits au lieu de 16 bits.


In [4]:
model = whisper.load_model("small")
print("Modèle Whisper chargé.")


Modèle Whisper chargé.


## 5. Enregistrement audio depuis le micro

Cette cellule enregistre la voix de l'utilisateur via le microphone.

Paramètres :
- `filename` : nom du fichier audio de sortie (`mon_audio.wav`).
- `duration` : durée de l'enregistrement (en secondes).
- `fs` : fréquence d'échantillonnage (16 kHz).

Le fichier est sauvegardé dans le **dossier courant** du notebook, ici :
`d:\\Desktop\\voix-langue-des-signes\\stt`

Après enregistrement, nous affichons à nouveau la liste des fichiers pour vérifier la présence de `mon_audio.wav`.


In [5]:
def record_audio(filename="mon_audio.wav", duration=5, fs=16000):
    print("Dossier courant :", os.getcwd())
    print("Enregistrement… Parle maintenant.")
    audio = sd.rec(int(duration * fs), samplerate=fs, channels=1, dtype="float32")
    sd.wait()
    audio_int16 = np.int16(audio * 32767)
    write(filename, fs, audio_int16)
    print("Enregistrement terminé :", filename)
    print("Fichiers présents :", os.listdir())

# Enregistrement d'un exemple
record_audio("mon_audio.wav", duration=5)


Dossier courant : d:\Desktop\voix-langue-des-signes\stt
Enregistrement… Parle maintenant.
Enregistrement terminé : mon_audio.wav
Fichiers présents : ['mon_audio.wav', 'stt_speech_to_text.ipynb']


## 6. Transcription de l'audio en texte

Cette cellule définit une fonction `speech_to_text` qui :

1. Vérifie que le fichier audio existe réellement (pour éviter les erreurs de type `FileNotFoundError`).
2. Appelle `model.transcribe(...)` de Whisper avec :
   - le chemin du fichier (`audio_path`),
   - la langue cible (`language="fr"`).
3. Retourne uniquement le champ `["text"]` (transcription finale).

Nous appliquons ensuite cette fonction sur `mon_audio.wav` pour afficher la transcription reconnue.


In [6]:
def speech_to_text(audio_path, language="fr"):
    # Vérification défensive : le fichier existe-t-il ?
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Le fichier {audio_path} est introuvable dans {os.getcwd()}")
    
    print("Transcription de :", audio_path)
    result = model.transcribe(audio_path, language=language)
    return result["text"]

# Test sur l'enregistrement précédent
texte = speech_to_text("mon_audio.wav")
print("Texte reconnu :", texte)


Transcription de : mon_audio.wav


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Texte reconnu :  Bonjour ça va ?


## 7. Récapitulatif détaillé : comment nous avons fait fonctionner la transcription

Cette section résume les étapes nécessaires pour faire fonctionner la transcription *parole → texte* avec Whisper dans notre environnement Windows + VS Code.

1. **Installation de ffmpeg**
   - Téléchargement de l’archive `ffmpeg-release-essentials.zip` depuis le site : https://www.gyan.dev/ffmpeg/builds/
   - Extraction du contenu dans `C:\ffmpeg\ffmpeg-8.0-essentials_build\`.
   - Vérification de la présence de `ffmpeg.exe` dans `C:\ffmpeg\ffmpeg-8.0-essentials_build\bin`.

2. **Ajout de ffmpeg au PATH**
   - Ajout de `C:\ffmpeg\ffmpeg-8.0-essentials_build\bin` dans la variable d’environnement `PATH` de Windows.
   - Vérification dans PowerShell que `ffmpeg` est bien reconnu via :
     ```powershell
     ffmpeg -version
     ```

3. **Problème rencontré dans le notebook VS Code**
   - Dans le kernel Python du notebook, appel à Whisper → erreur :
     `FileNotFoundError: [WinError 2] Le fichier spécifié est introuvable`.
   - Ce message ne concernait pas le `.wav` (qui existait bien dans le dossier du notebook), mais l'exécutable `ffmpeg` qui n'était pas trouvé par le **processus Python**.

4. **Correction dans le notebook**
   - Ajout explicite de `C:\ffmpeg\ffmpeg-8.0-essentials_build\bin` au `PATH` du processus dans le notebook :
     ```python
     os.environ["PATH"] += r";C:\ffmpeg\ffmpeg-8.0-essentials_build\bin"
     ```
   - Vérification que Python trouve bien `ffmpeg` :
     ```python
     import shutil
     print(shutil.which("ffmpeg"))
     ```
   - Une fois cela corrigé, Whisper a pu lire le fichier `mon_audio.wav` et la transcription a fonctionné.

5. **Pipeline final fonctionnel**
   - Enregistrement audio avec `sounddevice` dans `mon_audio.wav`.
   - Chargement du modèle Whisper `small`.
   - Appel de `model.transcribe("mon_audio.wav", language="fr")`.
   - Récupération de la transcription via `result["text"]`.

Conclusion :  
Nous disposons désormais d'un module STT fiable, capable de transformer un enregistrement audio micro en texte, et intégrable à la suite du pipeline (*texte → gloss → animations en langue des signes*).


## Version "semi temps réel" avec segments de 2 secondes

Objectif :
- Enregistrer la voix en continu, par segments de 2 secondes.
- Transcrire chaque segment dès qu'il est capturé.
- Afficher progressivement la transcription (segment par segment).
- Préparer un point d'accroche pour envoyer le texte vers le module "Texte → Signes".

Remarque :
- On n'enregistre plus dans un fichier `.wav` à chaque fois : on passe directement 
  le tableau NumPy au modèle Whisper, ce qui réduit les dépendances à `ffmpeg`.
- Cette approche est "semi temps réel" : latence ≈ durée segment (2 s) + temps de calcul.


In [7]:
import os
import shutil
import numpy as np
import sounddevice as sd
import whisper

# (Optionnel) S'assurer que ffmpeg est dans le PATH, si un jour tu veux aussi transcrire des fichiers
os.environ["PATH"] += r";C:\ffmpeg\ffmpeg-8.0-essentials_build\bin"
print("ffmpeg trouvé par Python ? ->", shutil.which("ffmpeg"))

# Paramètres audio
FS = 16000           # fréquence d'échantillonnage
SEGMENT_DURATION = 2 # durée d'un segment en secondes
N_SAMPLES = FS * SEGMENT_DURATION

# Chargement du modèle Whisper
# Tu peux tester "base" (plus rapide) ou rester sur "small"
model = whisper.load_model("small")
print("Modèle Whisper chargé.")


ffmpeg trouvé par Python ? -> C:\ffmpeg\ffmpeg-8.0-essentials_build\bin\ffmpeg.EXE
Modèle Whisper chargé.


## Boucle de capture et transcription par segments de 2 s

Stratégie :
- On utilise `sounddevice.rec` pour enregistrer un bloc de `SEGMENT_DURATION` secondes.
- On obtient un tableau NumPy `audio` de taille `(N_SAMPLES, 1)`, en float32, à 16 kHz.
- On passe ce tableau directement à `model.transcribe`, sans sauvegarde disque.
- On applique un seuil d'énergie simple pour ignorer (optionnellement) les segments silencieux.

Arrêt :
- La boucle tourne jusqu'à une interruption clavier (`Ctrl+C`).


In [8]:
def capture_and_transcribe_loop(language="fr", energy_threshold=0.01):
    """
    Enregistre et transcrit en continu, par segments de 2 secondes.
    
    - language : langue pour Whisper (ex: "fr")
    - energy_threshold : seuil moyen absolu sous lequel on considère le segment comme "silencieux"
    """
    print("Démarrage de la boucle semi temps réel.")
    print("Parle par petites phrases. Ctrl+C pour arrêter.\n")
    
    segment_idx = 0
    try:
        while True:
            print(f"\n[Segment {segment_idx}] Enregistrement…")
            audio = sd.rec(int(N_SAMPLES), samplerate=FS, channels=1, dtype="float32")
            sd.wait()  # on attend la fin du segment

            # On écrase en 1D (mono)
            audio_mono = audio[:, 0]

            # Mesure simple de l'énergie moyenne pour détecter le silence
            energy = float(np.mean(np.abs(audio_mono)))
            print(f"Énergie moyenne du segment : {energy:.5f}")

            if energy < energy_threshold:
                print("[Segment ignoré : trop silencieux]")
                segment_idx += 1
                continue

            # Transcription avec Whisper (sans fp16 sur CPU)
            print("[Transcription en cours…]")
            result = model.transcribe(audio_mono, language=language, fp16=False)
            text = result["text"].strip()

            if text:
                print(f"[Segment {segment_idx}] Texte reconnu : {text}")
                # 👉 ICI tu pourras appeler le module "texte -> gloss -> signes"
                # par ex: send_to_sign_module(text)
            else:
                print(f"[Segment {segment_idx}] Aucun texte reconnu.")

            segment_idx += 1

    except KeyboardInterrupt:
        print("\nArrêt de la boucle par l'utilisateur (Ctrl+C).")


## Lancement de la démo semi temps réel

Exécution :
- Lancer la cellule suivante.
- Parler par petites phrases (2–3 secondes max).
- Attendre que le texte apparaisse segment par segment.
- Pour arrêter : utiliser `Ctrl + C` dans le terminal / kernel.

Note :
- Plus le modèle est gros, plus la latence de transcription sera longue.
- Si c'est trop lent, tu peux tester `whisper.load_model("base")` ou `whisper.load_model("tiny")`.


In [11]:
capture_and_transcribe_loop(language="fr", energy_threshold=0.01)


Démarrage de la boucle semi temps réel.
Parle par petites phrases. Ctrl+C pour arrêter.


[Segment 0] Enregistrement…
Énergie moyenne du segment : 0.01627
[Transcription en cours…]
[Segment 0] Texte reconnu : et le bonjour

[Segment 1] Enregistrement…
Énergie moyenne du segment : 0.01706
[Transcription en cours…]
[Segment 1] Texte reconnu : J' bisher use dialecticale

[Segment 2] Enregistrement…
Énergie moyenne du segment : 0.00081
[Segment ignoré : trop silencieux]

[Segment 3] Enregistrement…
Énergie moyenne du segment : 0.01629
[Transcription en cours…]
[Segment 3] Texte reconnu : Je suis...

[Segment 4] Enregistrement…
Énergie moyenne du segment : 0.00266
[Segment ignoré : trop silencieux]

[Segment 5] Enregistrement…
Énergie moyenne du segment : 0.02440
[Transcription en cours…]
[Segment 5] Texte reconnu : Je suis... Je suis...

[Segment 6] Enregistrement…
Énergie moyenne du segment : 0.00187
[Segment ignoré : trop silencieux]

[Segment 7] Enregistrement…
Énergie moyenne du segmen

## 6. Boucle semi temps réel avec arrêt vocal

Cette cellule implémente une boucle de reconnaissance vocale **quasi temps réel**, 
basée sur un découpage du signal en segments de 2 secondes.  
Chaque segment est :

1. enregistré depuis le microphone,
2. analysé pour détecter le niveau d’énergie (afin d’ignorer les segments silencieux),
3. transcrit en texte à l’aide du modèle Whisper (`small`, `base` ou `tiny` selon les besoins),
4. affiché immédiatement.

### 🎯 Objectif
Obtenir un comportement proche du temps réel, où l’utilisateur parle par petites phrases
et la transcription s’affiche au fur et à mesure, avec une latence minimale.

### 🛑 Arrêt vocal naturel : dire **"stop"**
Dans un notebook VS Code, il est impossible de capturer des touches clavier comme `ESC` ou
d’utiliser des widgets interactifs de façon fiable.

Pour contourner cette limitation tout en restant cohérent avec le projet *parole → signes*,
l'arrêt de la boucle se fait via une **commande vocale** :

- si le segment transcrit contient le mot **"stop"** (majuscules ou minuscules),
- la boucle se termine automatiquement et proprement.

Exemple :  
> Vous dites "stop" → transcription → détection → arrêt immédiat.

### 🔍 Avantages de cette approche
- fonctionnement garanti dans 100% des notebooks,
- cohérence naturelle avec un système piloté par la voix,
- pas besoin de touches clavier ou de boutons graphiques,
- code robuste et facile à comprendre,
- latence faible et prévisible (≈ 2 à 4 secondes),
- parfait pour une démonstration ou un prototype.

### ⚠️ Remarque technique
Le segment est considéré comme silence si son énergie moyenne est inférieure
au seuil `energy_threshold`. Cela évite de solliciter Whisper inutilement.

Ci-dessous, le code complet de la boucle semi temps réel.


In [34]:
def capture_and_transcribe_loop(language="fr", energy_threshold=0.01):
    """
    Enregistre et transcrit en continu, par segments de 2 secondes.
    Arrêt vocal : dire 'stop'.
    """
    print("Démarrage de la boucle semi temps réel.")
    print("Parle par petites phrases. Pour arrêter, dis simplement 'stop'.\n")
    
    segment_idx = 0

    try:
        while True:
            print(f"\n[Segment {segment_idx}] Enregistrement…")

            audio = sd.rec(int(N_SAMPLES), samplerate=FS, channels=1, dtype="float32")
            sd.wait()

            audio_mono = audio[:, 0]
            energy = float(np.mean(np.abs(audio_mono)))
            print(f"Énergie moyenne du segment : {energy:.5f}")

            if energy < energy_threshold:
                print("[Segment ignoré : trop silencieux]")
                segment_idx += 1
                continue

            print("[Transcription en cours…]")
            result = model.transcribe(audio_mono, language=language, fp16=False)
            text = (result.get("text") or "").strip()

            if text:
                print(f"[Segment {segment_idx}] Texte reconnu : {text}")

                # 👉 ARRÊT vocal
                if "stop" in text.lower():
                    print("Commande vocale 'stop' détectée. Arrêt du système.")
                    break

            else:
                print(f"[Segment {segment_idx}] Aucun texte reconnu.")

            segment_idx += 1

    except KeyboardInterrupt:
        print("\nArrêt demandé par l'utilisateur (Ctrl+C ou bouton Stop).")

    print("Système arrêté proprement.")


## Lancer la démo semi temps réel 


In [36]:
capture_and_transcribe_loop(language="fr", energy_threshold=0.01)


Démarrage de la boucle semi temps réel.
Parle par petites phrases. Pour arrêter, dis simplement 'stop'.


[Segment 0] Enregistrement…
Énergie moyenne du segment : 0.02104
[Transcription en cours…]
[Segment 0] Texte reconnu : Bonjour

[Segment 1] Enregistrement…
Énergie moyenne du segment : 0.03690
[Transcription en cours…]
[Segment 1] Texte reconnu : Comment tu vas ?

[Segment 2] Enregistrement…
Énergie moyenne du segment : 0.02270
[Transcription en cours…]
[Segment 2] Texte reconnu : Bien et toi !

[Segment 3] Enregistrement…
Énergie moyenne du segment : 0.02013
[Transcription en cours…]
[Segment 3] Texte reconnu : Je vais super...

[Segment 4] Enregistrement…
Énergie moyenne du segment : 0.01280
[Transcription en cours…]
[Segment 4] Aucun texte reconnu.

[Segment 5] Enregistrement…
Énergie moyenne du segment : 0.03457
[Transcription en cours…]
[Segment 5] Texte reconnu : T'as mangé

[Segment 6] Enregistrement…
Énergie moyenne du segment : 0.01507
[Transcription en cours…]
[Segment 6]

## Section : Mise à jour du notebook pour la documentation STT
Ajout de textes explicatifs pour la partie semi temps réel.
